In [ ]:
import json
import numpy as np
import h5py
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
SPLITS_PATH = "/code/jjiang23/pathml/aim2_balanceV2/data/splits.json"
H5_KEY      = "world_mp_cropped_iou"   # MPW features
PHASE_NAMES = ["Phase1", "Phase2", "Phase3", "Phase4"]

LABEL_MAP = {
    b"Phase1": 0,
    b"Phase2": 1,
    b"Phase3": 2,
    b"Phase4": 3,
    b"nonphase": 4,   # excluded
}

XGB_PARAMS = dict(
    objective        = "multi:softprob",
    num_class        = 4,
    n_estimators     = 400,
    max_depth        = 6,
    learning_rate    = 0.1,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    use_label_encoder= False,
    eval_metric      = "mlogloss",
    tree_method      = "hist",
    verbosity        = 0,
    n_jobs           = -1,
    random_state     = 42,
)

with open(SPLITS_PATH) as f:
    splits = json.load(f)

print(f"Folds: {list(splits.keys())}")

In [ ]:
def load_phase_frames(h5_paths):
    """
    Load per-frame skeleton features and phase labels (0-3) from a list of h5 files.
    Nonphase frames (label==4) are excluded.

    Returns:
        X: (N, J*3) float32  — flattened per-frame keypoints
        y: (N,)   int        — phase label 0-3
    """
    Xs, ys = [], []
    for path in h5_paths:
        try:
            with h5py.File(path, 'r') as f:
                # (T, 1, J, >=3) → (T, J, 3)
                kp = f[H5_KEY][:][:, 0, :, :3].astype(np.float32)  # (T, J, 3)
                raw_labels = f['camera_poses_labels'][:]
        except Exception as e:
            print(f"  Skipping {path}: {e}")
            continue

        labels = np.array([LABEL_MAP[lbl] for lbl in raw_labels], dtype=np.int32)
        kp     = np.nan_to_num(kp)

        # Keep only phase frames (0-3)
        mask = labels < 4
        if mask.sum() == 0:
            continue

        T, J, D = kp[mask].shape
        Xs.append(kp[mask].reshape(T, J * D))   # (T, J*3)
        ys.append(labels[mask])                  # (T,)

    if not Xs:
        return np.empty((0, 0), dtype=np.float32), np.empty((0,), dtype=np.int32)
    return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0)


# Quick sanity check on fold 0
fold0 = splits['fold_0']
X_s, y_s = load_phase_frames(fold0['train'][:3])
print(f"Sample X shape: {X_s.shape}, y distribution: {np.bincount(y_s)}")

In [ ]:
# ── Train + evaluate across all 5 folds ─────────────────────────────────────
fold_results = []

for fold_name, fold_data in splits.items():
    print(f"\n{'='*60}")
    print(f"  {fold_name}")
    print(f"{'='*60}")

    print("  Loading train...", end=" ")
    X_train, y_train = load_phase_frames(fold_data['train'])
    print(f"{X_train.shape[0]:,} phase frames | dist: {np.bincount(y_train)}")

    print("  Loading val...",   end=" ")
    X_val,   y_val   = load_phase_frames(fold_data['val'])
    print(f"{X_val.shape[0]:,} phase frames   | dist: {np.bincount(y_val)}")

    if X_train.shape[0] == 0 or X_val.shape[0] == 0:
        print("  Skipping fold — empty split")
        continue

    clf = xgb.XGBClassifier(**XGB_PARAMS)
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    y_pred = clf.predict(X_val)
    acc    = accuracy_score(y_val, y_pred)

    print(f"  Val Accuracy: {acc:.4f}")
    print(classification_report(y_val, y_pred, target_names=PHASE_NAMES, digits=3))

    fold_results.append({
        "fold":       fold_name,
        "model":      clf,
        "acc":        acc,
        "y_val":      y_val,
        "y_pred":     y_pred,
        "cm":         confusion_matrix(y_val, y_pred),
    })

In [ ]:
# ── Summary across folds ─────────────────────────────────────────────────────
accs = [r['acc'] for r in fold_results]
print(f"\nAccuracy across {len(accs)} folds:")
for r in fold_results:
    print(f"  {r['fold']}: {r['acc']:.4f}")
print(f"  Mean ± Std: {np.mean(accs):.4f} ± {np.std(accs):.4f}")

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
n_folds = len(fold_results)
fig, axes = plt.subplots(1, n_folds, figsize=(5 * n_folds, 4))
if n_folds == 1:
    axes = [axes]

for ax, r in zip(axes, fold_results):
    cm_norm = r['cm'].astype(float) / r['cm'].sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt=".2f", ax=ax,
        xticklabels=PHASE_NAMES, yticklabels=PHASE_NAMES,
        cmap="Blues", vmin=0, vmax=1, cbar=False,
    )
    ax.set_title(f"{r['fold']}  acc={r['acc']:.3f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.suptitle("Phase Classification — XGBoost on MPW Skeleton (nonphase excluded)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pooled confusion matrix (all folds combined) ─────────────────────────────
y_val_all  = np.concatenate([r['y_val']  for r in fold_results])
y_pred_all = np.concatenate([r['y_pred'] for r in fold_results])

cm_all = confusion_matrix(y_val_all, y_pred_all)
cm_norm_all = cm_all.astype(float) / cm_all.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_norm_all, annot=True, fmt=".2f", ax=ax,
    xticklabels=PHASE_NAMES, yticklabels=PHASE_NAMES,
    cmap="Blues", vmin=0, vmax=1,
)
ax.set_title(f"Pooled — Acc={accuracy_score(y_val_all, y_pred_all):.4f}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.show()

print(classification_report(y_val_all, y_pred_all, target_names=PHASE_NAMES, digits=3))